# Arbora MiniC training (Colab)

Two independent CodeT5-small tasks: syntax repair and AST linearization. The TypeScript parser remains the validity oracle. No result is claimed until its training and evaluation cell completes. On a Colab T4, the full 50k / four-epoch run is expected to take multiple hours; this is an unmeasured planning estimate. Quick mode uses 5k generated programs and one epoch and targets under 10 minutes, but downloads and runtime load can exceed that.

In [ ]:
# Read secrets from Colab Secrets; do not print them or mount Drive.
import os, time, subprocess, sys, json, platform
from google.colab import userdata
started_at = time.time()
hf_token = userdata.get('HF_TOKEN', default='')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
# Optional Gemini credentials are read from Colab Secrets only in the Gemini cell.
QUICK_MODE = False  # True: 5,000 programs, one epoch; target <10 min, not guaranteed.
N_PROGRAMS = 5_000 if QUICK_MODE else 50_000
print('Runtime:', platform.python_version(), platform.platform())
print('Mode:', 'quick' if QUICK_MODE else 'full', '| generated programs:', N_PROGRAMS)

## Install runtime and clone

Colab Secrets are not written into notebooks or shell history. The public repository clone contains code only; dataset, checkpoints and private result caches remain in the runtime.

In [ ]:
!apt-get update -qq && apt-get install -y -qq curl
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
!apt-get install -y -qq nodejs
%cd /content
!git clone https://github.com/mannbutani0306/arbora.git
%cd /content/arbora
!npm ci
!python -m pip install -q -r ml/requirements.txtprint(subprocess.check_output(['node', '--version'], text=True).strip())

## Generate, split, corrupt, and check for leakage

The split is by program ID with seed 42. The TypeScript injector only retains corruptions that produce parser errors. Preparation reports source-token and model-token filtering; no sequence is silently truncated.

In [ ]:
!npx tsx tools/generate-dataset.ts --n {N_PROGRAMS} --seed 42 --out ml/data/raw.jsonl
!npx tsx tools/split-dataset.ts --in ml/data/raw.jsonl --seed 42 --out-dir ml/data
!PYTHONPATH=ml/src python ml/scripts/prepare_data.py --repo-root . --data-dir ml/data --per-program 3
!PYTHONPATH=ml/src python ml/scripts/filter_model_tokens.py --data-dir ml/data --split train
!PYTHONPATH=ml/src python ml/scripts/filter_model_tokens.py --data-dir ml/data --split val
!PYTHONPATH=ml/src python ml/scripts/filter_model_tokens.py --data-dir ml/data --split test
!PYTHONPATH=ml/src python ml/scripts/check_data_leakage.py --data-dir ml/data

## Train both tasks

Each task uses its own model and best checkpoint. Resume with `--resume` (or `--resume /content/arbora/ml/runs/repair-t5/checkpoint-N`) after an interrupted run.

In [ ]:
quick_flag = ' --quick' if QUICK_MODE else ''
!PYTHONPATH=ml/src python -m arbora_ml.train --config ml/configs/repair_t5.yaml{quick_flag}
!PYTHONPATH=ml/src python -m arbora_ml.train --config ml/configs/ast_t5.yaml{quick_flag}

## Baselines and neural evaluation

All repair baselines, Gemini (when enabled), and Arbora-T5 use the same deterministic 300 corrupted test IDs, selected by SHA-256 of `42:<id>`. The full held-out split remains available in JSONL for larger evaluations. B1/B2 invoke the deterministic TypeScript repair implementation and may take several minutes.

In [ ]:
!npx tsx tools/run-baselines.ts --train ml/data/train.jsonl --test ml/data/test.jsonl --corrupted ml/data/test-corrupted-model-filtered.jsonl --limit 300 --seed 42 --out ml/results/baselines.json
!PYTHONPATH=ml/src python ml/scripts/score_baselines.py --in ml/results/baselines.observations.jsonl --out ml/results/evaluation_baselines.json
!PYTHONPATH=ml/src python -m arbora_ml.evaluate --task repair --model ml/runs/repair-t5/best --test ml/data/test.jsonl --corrupted ml/data/test-corrupted-model-filtered.jsonl --limit 300 --seed 42 --out ml/results/evaluation_repair.json --repo-root .
!PYTHONPATH=ml/src python -m arbora_ml.evaluate --task ast --model ml/runs/ast-t5/best --test ml/data/test-ast-model-filtered.jsonl --limit 300 --seed 42 --out ml/results/evaluation_ast.json --repo-root .

## Optional Gemini and OOD evaluations

Gemini secrets are optional and requested only if you elect to run the fixed-prompt, temperature-zero baseline. OOD requires Supabase to be configured for exactly 12 valid sample programs; the exporter fails closed if those records are unavailable.

In [ ]:
run_gemini = False
if run_gemini:
    gemini_key = userdata.get('GEMINI_API_KEY', default='')
    gemini_model = userdata.get('GEMINI_MODEL', default='')
    if not gemini_key or not gemini_model:
        raise RuntimeError('Configure GEMINI_API_KEY and GEMINI_MODEL in Colab Secrets first.')
    os.environ['GEMINI_API_KEY'] = gemini_key
    os.environ['GEMINI_MODEL'] = gemini_model
    !PYTHONPATH=ml/src python ml/scripts/gemini_baseline.py --in ml/data/test-corrupted-model-filtered.jsonl --limit 300 --seed 42
    !PYTHONPATH=ml/src python ml/scripts/score_gemini.py --cache ml/results/gemini_repair_cache.jsonl --test ml/data/test.jsonl --repo-root .
    del os.environ['GEMINI_API_KEY']
    del gemini_key
# Run separately after configuring Supabase sample access:
# !npx tsx tools/export-ood.ts ml/data/ood.jsonl

## Plots, summary, optional Hub publish, and download

Hub repository names are supplied via runtime environment variables, never embedded in source. The Hub upload defaults to private. No Supabase credentials are stored in this notebook.

In [ ]:
!PYTHONPATH=ml/src python -m arbora_ml.plots
!PYTHONPATH=ml/src python ml/scripts/generate_summary.py --results ml/results --out ml/results/summary.md
publish = False
if publish and hf_token:
    for task in ('repair', 'ast'):
        if os.environ.get(f'HF_{task.upper()}_REPO_ID'):
            !PYTHONPATH=ml/src python ml/scripts/push_to_hub.py --task {task} --checkpoint ml/runs/{task}-t5/best
import shutil
shutil.make_archive('/content/arbora-ml-results', 'zip', 'ml/results')
from google.colab import files
files.download('/content/arbora-ml-results.zip')
print('Elapsed minutes:', round((time.time() - started_at) / 60, 1))